# 简介

模仿学习（Imitation Learning）是一种机器学习方法，智能体通过学习来模仿专家的动作。要想训练出一个鲁棒（robust）的智能体，需要大量的数据。然而，通过人工演示手动采集数据既耗时又昂贵。

Isaac Lab Mimic 是 Isaac Lab 中内置的一项功能，它允许用户基于少量人工演示，通过合成轨迹来生成新的演示数据。本蓝图（Blueprint）将展示如何使用 Isaac Lab Mimic 为 Franka 机械臂生成新的运动轨迹，然后使用 NVIDIA Cosmos 进行视觉增强，从而创建用于模仿学习的数据集。整个工作流程分为两个主要步骤：

1. 使用 Isaac Lab Mimic，基于少量人工演示合成轨迹，生成新的演示数据。
2. 使用 NVIDIA Cosmos 对新生成的演示数据应用多样化的视觉变换，创建规模大、多样性强的数据集。

本 Notebook 将逐步引导你完成整个工作流程。

**注意：本 Notebook 必须在与 Isaac Sim 模拟器相同的机器上运行，且该机器必须连接显示器。**


# 理解本蓝图

## 运动轨迹合成
Isaac Lab Mimic 是 Isaac Lab 附带的一组功能集（Isaac Lab 是一个旨在帮助训练机器人策略的开源机器人学习框架）。Mimic 的核心理念是：让用户仅需少量人工演示，就能合成大量新的机器人运动轨迹，从而大大减少为模仿学习采集数据集所需的时间和精力。

人工演示数据带有子任务（subtask）标注信息，Isaac Lab Mimic 利用这些标注，通过对原始演示进行空间变换，为新场景配置构建运动轨迹。

## 视觉增强
新的运动轨迹生成后，可以使用 NVIDIA Cosmos 对其进行视觉增强，创建适合训练模仿学习策略的多样化数据集。

通过这种多阶段的数据生成方案，我们无需大量人工数据，即可自动创建用于训练复杂模仿学习策略的鲁棒数据集，大幅增加可用于训练的数据量，同时缩短采集数据集所需的时间。


# 生成新的运动轨迹



## 设置 Isaac Lab 初始配置

此单元格用于设置数据生成的基础配置：

1. **如何修改**：
   - 根据你的 GPU 性能调整 `num_envs`
   - 将 `generation_num_trials` 设置为希望执行的成功试验（trial）次数。注意部分试验可能不成功，因此实际执行的总试验次数可能会更多。

2. **提示**：
   - 测试时先设为 1 次试验，正式训练时再增加。增加试验次数会延长数据集生成所需的时间。


In [ ]:
from notebook_widgets import create_num_trials_input

num_envs = 1
num_trials = create_num_trials_input()


## 启动仿真

运行此单元格以启动仿真环境。它将为数据生成搭建必要的组件。

**注意**：仿真正在运行时，可能会弹出 **"Isaac Sim 未响应"** 的提示窗口。这属于正常现象，请点击 **Wait（等待）** 选项，耐心等待进程执行完毕。

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()

from argparse import ArgumentParser, Namespace
from isaaclab.app import AppLauncher

parser = ArgumentParser()
AppLauncher.add_app_launcher_args(parser)
args_cli = parser.parse_args([])
args_cli.enable_cameras = True
args_cli.kit_args = "--enable omni.videoencoding"

config = {
    "task": "Isaac-Stack-Cube-Franka-IK-Rel-Blueprint-Mimic-v0",  
    "num_envs": num_envs,                                       
    "generation_num_trials": num_trials.value,                         
    "input_file": "datasets/annotated_dataset.hdf5",     
    "output_file": "datasets/generated_dataset.hdf5", 
    "pause_subtask": False,
    "enable": "omni.kit.renderer.capture",
}

# Update the default configuration
args_dict = vars(args_cli)
args_dict.update(config)
args_cli = Namespace(**args_dict)

# Now launch the simulator with the final configuration
app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app

import asyncio
import gymnasium as gym
import numpy as np
import random
import torch

import isaaclab_mimic.envs  # noqa: F401
from isaaclab_mimic.datagen.generation import env_loop, setup_env_config, setup_async_generation
from isaaclab_mimic.datagen.utils import get_env_name_from_dataset, setup_output_paths, interactive_update_randomizable_params, reset_env
from isaaclab.managers import ObservationTermCfg as ObsTerm
from notebook_utils import ISAACLAB_OUTPUT_DIR

import isaaclab_tasks  # noqa: F401
num_envs = args_cli.num_envs

# Setup output paths and get env name
output_dir, output_file_name = setup_output_paths(args_cli.output_file)
env_name = args_cli.task or get_env_name_from_dataset(args_cli.input_file)

# Configure environment
env_cfg, success_term = setup_env_config(
    env_name=env_name,
    output_dir=output_dir,
    output_file_name=output_file_name,
    num_envs=num_envs,
    device=args_cli.device,
    generation_num_trials=args_cli.generation_num_trials,
)
# Set observation output directory
for obs in vars(env_cfg.observations.rgb_camera).values():
    if not isinstance(obs, ObsTerm):
        continue
    obs.params["image_path"] = os.path.join(ISAACLAB_OUTPUT_DIR, obs.params["image_path"])
env_cfg.observations


# create environment
env = gym.make(env_name, cfg=env_cfg).unwrapped

# set seed for generation
random.seed(env.cfg.datagen_config.seed)
np.random.seed(env.cfg.datagen_config.seed)
torch.manual_seed(env.cfg.datagen_config.seed)

# reset before starting
reset_env(env, 100)


## 交互式参数更新

为了让生成的运动轨迹具有多样性，每次试验都会对场景配置进行随机化。本节提供交互式滑块和控件，可实时调整各种环境参数：

1. **你将看到的内容**：
   - 数值滑块
   - 用于最小值/最大值设置的范围输入框
   - 当前值的显示
   - 参数名称及其允许的取值范围

2. **使用方法**：
   - 拖动滑块调整数值
   - 观察环境的实时更新

3. **可用参数**：
   - **Franka 关节状态随机化**：
     - **mean (0.0 - 0.5)**：控制关节角度偏移的平均值（单位：弧度）
     - **std (0.0 - 0.1)**：控制随机化围绕均值的散布程度

   - **立方体位置随机化**：
     - **pose_range.x (0.3 - 0.9)**：控制立方体沿 x 轴的放置位置（单位：米）
     - **pose_range.y (-0.3 - 0.3)**：控制立方体沿 y 轴的放置位置（单位：米）
     - **min_separation (0.0 - 0.5)**：立方体之间允许的最小间距（单位：米）
     
     **注意：**如果由于空间限制，系统经过多次尝试仍无法按指定的最小间距放置立方体，它将接受最后生成的位置，即使该位置不满足间距要求。这样可以避免系统卡在无法满足的配置上。


4. **提示**：
   - 建议先做小幅调整，以了解各个参数的效果

注意：这些调整会影响新演示数据的生成方式，建议多尝试不同的设置组合，以达到理想的效果。

In [ ]:
randomizable_params = {
    "randomize_franka_joint_state": {
        "mean": (0.0, 0.5, 0.01),
        "std": (0.0, 0.1, 0.01),
    },
    "randomize_cube_positions": {
        "pose_range": {
                "x": (0.3, 0.9, 0.01),
                "y": (-0.3, 0.3, 0.01),
            },
        "min_separation": (0.0, 0.5, 0.01),
    }
}

for i in range(len(env.unwrapped.event_manager._mode_term_cfgs["reset"])):
    event_term = env.unwrapped.event_manager._mode_term_cfgs["reset"][i]
    name = env.unwrapped.event_manager.active_terms["reset"][i]
    display(f"Updating parameters for event: {event_term.func.__name__}")
    interactive_update_randomizable_params(event_term, name, randomizable_params[name], env=env)


## 数据生成

运行此单元格，使用你已配置好的参数开始生成演示数据。该过程将会：
- 生成指定数量的演示数据
- 将成功的演示保存到输出文件中
- 在生成过程中显示进度

In [ ]:
import sys
from IPython.display import display, Pretty

# Create a new output capture
class OutputCapture:
    def __init__(self):
        self._buffer = ""
    
    def write(self, text):
        if text.strip():  # Only process non-empty strings
            display(Pretty(text.rstrip()))
    
    def flush(self):
        if self._buffer:
            display(Pretty(self._buffer))
            self._buffer = ""

# Move stdout redirection before setup_async_generation
old_stdout = sys.stdout
sys.stdout = OutputCapture()

try:
    # Setup and run async data generation
    async_components = setup_async_generation(
        env=env,
        num_envs=args_cli.num_envs,
        input_file=args_cli.input_file,
        success_term=success_term,
        pause_subtask=args_cli.pause_subtask
    )

    future = asyncio.ensure_future(asyncio.gather(*async_components['tasks']))
    env_loop(env, async_components['action_queue'], 
            async_components['info_pool'], async_components['event_loop'])
except asyncio.CancelledError:
    display(Pretty("Tasks were cancelled."))
except AttributeError as e:
    if "'FrankaCubeStackIKRelMimicEnv' object has no attribute 'scene'" in str(e):
        display(Pretty("Environment was closed during execution. This is expected behavior."))
except Exception as e:
    display(Pretty(f"Error occurred: {str(e)}"))
finally:
    # Restore original stdout first
    sys.stdout = old_stdout
    
    # Cancel the future and ignore any AttributeErrors from pending tasks
    if 'future' in locals():
        future.cancel()
        try:
            async_components['event_loop'].run_until_complete(future)
        except (asyncio.CancelledError, AttributeError) as e:
            if isinstance(e, AttributeError) and "'FrankaCubeStackIKRelMimicEnv' object has no attribute 'scene'" in str(e):
                display(Pretty("Environment was closed during execution. This is expected behavior!"))
            elif isinstance(e, asyncio.CancelledError):
                display(Pretty("Tasks were properly cancelled during cleanup."))
            else:
                display(Pretty(f"Unexpected cleanup error: {str(e)}"))


# Cosmos

至此，新的运动轨迹已生成完毕。接下来我们将使用 Cosmos 对数据进行视觉变换，创建适合训练模仿学习策略的逼真演示数据。

## 视频预处理
第一步，我们将把生成的运动轨迹处理成视频，作为 Cosmos 模型的输入。
这里利用场景的法线（normals）为语义分割图像添加着色效果，由此得到的输入与 Cosmos 模型的配合效果非常好。

In [ ]:
from notebook_widgets import create_camera_input
from notebook_utils import ISAACLAB_OUTPUT_DIR

VIDEO_LENGTH = 120   # Suggested length is between 120 and 200
camera_selection = create_camera_input(ISAACLAB_OUTPUT_DIR)


In [ ]:
import os
from IPython.display import Video
from notebook_utils import encode_video, ISAACLAB_OUTPUT_DIR, get_env_trial_frames

env_trial_frames = get_env_trial_frames(ISAACLAB_OUTPUT_DIR, camera_selection.value, 10)
camera = camera_selection.value
for env_num, trial_nums in env_trial_frames.items():
    for trial_num, (start_frame, end_frame) in trial_nums.items():
        trial_length = end_frame - start_frame + 1
        if trial_length < VIDEO_LENGTH:
            print(f"\nSkipping Trial {trial_num}: Too short ({trial_length} frames)")
            continue
            
        video_start = max(start_frame, end_frame - VIDEO_LENGTH + 1)
        
        # Generate video filename with trial number
        video_filepath = os.path.join(ISAACLAB_OUTPUT_DIR, f"shaded_segmentation_{camera}_trial_{trial_num}_tile_{env_num}.mp4")
            
        try:
            encode_video(ISAACLAB_OUTPUT_DIR, video_start, VIDEO_LENGTH, 
                        camera, video_filepath, env_num, trial_num)
            display(video_filepath)
            display(Video(video_filepath, width=1000))
        except ValueError as e:
            print(f"Error processing trial {trial_num}: {str(e)}")


## 部署 Cosmos
你可以将 Cosmos 部署到自己选择的云服务商上，或者部署到本地资源：[Cosmos Transfer1](https://huggingface.co/nvidia/Cosmos-Transfer1-7B)。
点击 Cosmos Transfer 页面上的 `Code` 链接，按照 README 中的安装步骤操作即可。详细的设置说明可以在 `examples/inference_multi_control_manual_input.md` 中找到。

> ### 为 Cosmos Transfer1 添加 Web API
> 为了简化测试，请将文件 `notebook/app.py` 复制到 Cosmos 根目录下，并用 `python app.py` 运行。这将启动一些 HTTP 接口端点（endpoints），供 Notebook 与 Cosmos 模型之间通信使用。默认情况下，该脚本在端口 `5000` 上提供服务。

In [ ]:
import ipywidgets as widgets
url_widget = widgets.Text(value="", placeholder="cosmos/url:port", description="Cosmos URL:", style={'description_width': 'initial'}, layout={"width": "1000px"})
display(url_widget)


### 使用 Cosmos 模型

Cosmos 模型提供了多个可用参数，它们会以不同方式影响输出结果：
- `prompt`：用于视频生成的文本提示词。
- `seed`：随机数生成器的种子。`int [0 - 2147483648]`
- `control_weight`：控制控制输入对输出的影响强度。影响越强，输出越贴合控制输入，但模型生成的自由度越低。`float [0 - 1.0]`
- `sigma_max`：表示最大 sigma 的浮点数。取值越小，输出相对于原始输入的变化越小；取值越大，允许的变化越大，但输出可能会更加偏离输入场景。`float [0 - 80.0]`

In [ ]:
from notebook_widgets import create_variable_dropdowns, create_cosmos_params
from notebook_utils import ISAACLAB_OUTPUT_DIR, COSMOS_OUTPUT_DIR

prompt_manager = create_variable_dropdowns("stacking_prompt.toml")
cosmos_params = create_cosmos_params(ISAACLAB_OUTPUT_DIR)


## 使用 Cosmos 生成
---
> **注意：** 在单张 H100 GPU 上，视视频长度而定，生成过程通常耗时约 5 到 10 分钟。

---

> **提示：**
> - 如果想提高对提示词的遵循程度，可以尝试调大 `Sigma Max` 值
> - 如果想减少输出偏离输入场景的程度，可以尝试调大 `Control Weight` 和/或 `Canny Strength`

In [ ]:
import os
from cosmos_request import process_video
from notebook_utils import ISAACLAB_OUTPUT_DIR
from notebook_widgets import create_download_link
from IPython.display import Video, clear_output

params = {k: w.value for k, w in cosmos_params.items()}
video_filepath = os.path.join(ISAACLAB_OUTPUT_DIR, params.pop("input_video"))
output_path = f"{COSMOS_OUTPUT_DIR}/cosmos_{params['seed']}.mp4"
params["prompt"] = prompt_manager.prompt

if not url_widget.value:
    raise ValueError("Enter URL to proceed.")

response = process_video(
    url=url_widget.value,
    video_path=video_filepath,
    output_path=output_path,
    **params,
)
if response is None:
    display("An error occurred processing the request")
elif response.status_code == 200:
    clear_output(wait=True)
    display(Video(output_path))
    display(create_download_link(output_path, link_text=f"Download Video: {output_path}"))
